In [55]:
import pandas as pd 
import json
import numpy
import sys
from pathlib import Path
import re
import networkx as nx





In [56]:
sys.path.insert(0, r"..\LLM_Open_IE\scripts\graphs")
from graph_plot import GraphPlotter

In [57]:
import re
pasta = Path.cwd()

while not (pasta / "Outputs").exists() and pasta != pasta.parent:
    pasta = pasta.parent


arquivos = [
    arquivo
    for arquivo in (pasta / "Outputs").rglob("Respostas*/*metricas_sentencas.csv")
    if not any(
        re.fullmatch(r"k_\d+", parte.name, re.IGNORECASE)
        for parte in arquivo.parents
    )
]

dataframes = []

for arquivo in arquivos:
    df_temp = pd.read_csv(arquivo)
    df_temp["pasta_respostas"] = arquivo.parent.name
    dataframes.append(df_temp)

df = pd.concat(dataframes, ignore_index=True)
df["k"] = df["k"].fillna("")
df = df[df["k"] == ""]

print(df)

        corpus        lote k                       prompt  \
0      oiec_pt  10_batches     Respostas ExtrativoDPTO-IE   
1      oiec_pt  10_batches     Respostas ExtrativoDPTO-IE   
2      oiec_pt  10_batches     Respostas ExtrativoDPTO-IE   
3      oiec_pt  10_batches     Respostas ExtrativoDPTO-IE   
4      oiec_pt  10_batches     Respostas ExtrativoDPTO-IE   
...        ...         ... ..                         ...   
13939      bia  25_batches     Respostas ExtrativoOIEC-PT   
13940      bia  25_batches     Respostas ExtrativoOIEC-PT   
13941      bia  25_batches     Respostas ExtrativoOIEC-PT   
13942      bia  25_batches     Respostas ExtrativoOIEC-PT   
13943      bia  25_batches     Respostas ExtrativoOIEC-PT   

                     arquivo  sentenca  \
0             Claude_Sonnet5         0   
1             Claude_Sonnet5         1   
2             Claude_Sonnet5         2   
3             Claude_Sonnet5         3   
4             Claude_Sonnet5         4   
...            

In [58]:
T = lambda s: [(t["arg1"],t["rel"],t["arg2"]) for t in json.loads(s)] if isinstance(s,str) else []
IMG = Path(r"C:\Users\User\Documents\IC_FORMAS\raw\TASKS\TASK-1\LLM_Open_IE\Outputs\graphs\images\ciclos")

In [60]:
for (corpus,lote,prompt), g in df[df['prompt']!='gold'].groupby(["corpus",'lote','prompt']):
    gold = df[(df['corpus'] == corpus) & (df['prompt'] == 'gold')].drop_duplicates('frases').set_index('frase')['relacoes_json']
    plotter = GraphPlotter(str(IMG/corpus/lote/prompt))
    for i, (frase,gf) in enumerate(g[g['ciclos_dirigidos']>0].groupby('frase')):
        paineis = [('gold',T(gold.get('frase')))]
        paineis += [(f"{r.arquivo} (ciclo)",T(r.relacoes_json)) for r in gf.itertuples()]
        plotter.plot_triplas(paineis, frase, f"{i:03d}")